In [ ]:
# ============================================================
# Notebook 02 - Feature Extraction
# CS-419 Deep Learning Project - Speech Emotion Recognition
# ============================================================
# This notebook extracts and caches features for all 3 model types:
#   1. Flat MFCC features  (for MLP baseline)
#   2. Log-mel spectrograms (for CNN and CNN-LSTM)
# ============================================================

import sys
import os
import numpy as np
import time

sys.path.append("../src")

from data_loader import load_tess_paths, split_dataset
from feature_extraction import (
    extract_all_flat,
    extract_spectrogram,
    batch_extract_features,
)
from augmentation import generate_augmented_sample
from utils import set_seeds, load_or_compute

set_seeds(42)
os.makedirs("results/cache", exist_ok=True)

# ---- Cell 1: Load dataset paths ----

DATA_ROOT = "data/TESS Toronto emotional speech set data"
df = load_tess_paths(DATA_ROOT)
df_train, df_val, df_test = split_dataset(df)

print(f"Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")

# ---- Cell 2: Extract flat MFCC features (MLP input) ----
# Feature vector: 40 MFCC mean + 40 MFCC std + 12 chroma + 4 ZCR/RMS = 96 dims

print("\n--- Extracting flat MFCC features ---")

t0 = time.time()
X_train_flat, y_train = load_or_compute(
    "results/cache/train_flat.npz",
    batch_extract_features,
    df_train, extract_all_flat, "Train flat"
)
X_val_flat, y_val = load_or_compute(
    "results/cache/val_flat.npz",
    batch_extract_features,
    df_val, extract_all_flat, "Val flat"
)
X_test_flat, y_test = load_or_compute(
    "results/cache/test_flat.npz",
    batch_extract_features,
    df_test, extract_all_flat, "Test flat"
)
elapsed = time.time() - t0

print(f"\nFlat feature shapes:")
print(f"  X_train: {X_train_flat.shape}  y_train: {y_train.shape}")
print(f"  X_val  : {X_val_flat.shape}    y_val  : {y_val.shape}")
print(f"  X_test : {X_test_flat.shape}   y_test : {y_test.shape}")
print(f"  Time   : {elapsed:.1f}s")

# ---- Cell 3: Normalize flat features ----
# Zero-mean, unit-variance normalization using training statistics

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_flat_norm = scaler.fit_transform(X_train_flat)
X_val_flat_norm   = scaler.transform(X_val_flat)
X_test_flat_norm  = scaler.transform(X_test_flat)

# Save scaler for later use
import pickle
with open("results/cache/flat_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
print("Scaler saved to results/cache/flat_scaler.pkl")

# Verify no NaNs
assert not np.isnan(X_train_flat_norm).any(), "NaN found in training features!"
print("\nNormalization complete. No NaN values found.")

# ---- Cell 4: Extract log-mel spectrograms (CNN/LSTM input) ----
# Each spectrogram: (128, 128, 1) grayscale image

print("\n--- Extracting log-mel spectrograms ---")
print("This takes 3-5 minutes on CPU. On GPU (Colab) it is faster.")

t0 = time.time()
X_train_spec, _ = load_or_compute(
    "results/cache/train_spec.npz",
    batch_extract_features,
    df_train, extract_spectrogram, "Train spec"
)
X_val_spec, _ = load_or_compute(
    "results/cache/val_spec.npz",
    batch_extract_features,
    df_val, extract_spectrogram, "Val spec"
)
X_test_spec, _ = load_or_compute(
    "results/cache/test_spec.npz",
    batch_extract_features,
    df_test, extract_spectrogram, "Test spec"
)
elapsed = time.time() - t0

print(f"\nSpectrogram shapes:")
print(f"  X_train: {X_train_spec.shape}")
print(f"  X_val  : {X_val_spec.shape}")
print(f"  X_test : {X_test_spec.shape}")
print(f"  Time   : {elapsed:.1f}s")

# ---- Cell 5: Sanity check - visualize extracted spectrograms ----

import matplotlib.pyplot as plt
from data_loader import EMOTION_LABELS

plt.rcParams.update({"figure.dpi": 110})

n_show = 8
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
fig.suptitle("Extracted Spectrograms (from cache)", fontsize=12)
axes = axes.flatten()

for i in range(n_show):
    axes[i].imshow(X_train_spec[i, :, :, 0], aspect="auto",
                   origin="lower", cmap="magma")
    axes[i].set_title(EMOTION_LABELS[int(y_train[i])].title(), fontsize=9)
    axes[i].axis("off")

plt.tight_layout()
plt.savefig("results/plots/extracted_spectrograms_check.png", dpi=130, bbox_inches="tight")
plt.show()

# ---- Cell 6: Demonstrate augmentation ----

from augmentation import augment_waveform, augment_spectrogram
from feature_extraction import load_audio, extract_log_mel_spectrogram, SAMPLE_RATE

sample_path = df_train.iloc[0]["path"]
y_orig, sr  = load_audio(sample_path)
spec_orig   = extract_log_mel_spectrogram(y_orig, sr)

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.suptitle("Augmentation Examples (same file, different transforms)",
             fontsize=12, fontweight="bold")

aug_names = [
    "Original",
    "Noise added",
    "Time stretched",
    "Pitch shifted",
    "Time shifted",
    "Freq masked",
    "Time masked",
    "Noise + Pitch",
]

y_noise  = y_orig + 0.005 * np.random.randn(len(y_orig))
from augmentation import time_stretch, pitch_shift, time_shift, frequency_mask, time_mask

specs = [
    spec_orig,
    extract_log_mel_spectrogram(y_noise, sr),
    extract_log_mel_spectrogram(time_stretch(y_orig, rate=1.15), sr),
    extract_log_mel_spectrogram(pitch_shift(y_orig, sr, n_steps=3), sr),
    extract_log_mel_spectrogram(time_shift(y_orig, 0.15), sr),
    frequency_mask(spec_orig, max_width=15),
    time_mask(spec_orig, max_width=20),
    extract_log_mel_spectrogram(
        pitch_shift(y_noise, sr, n_steps=-2), sr
    ),
]

for ax, spec, name in zip(axes.flatten(), specs, aug_names):
    ax.imshow(spec[:, :, 0], aspect="auto", origin="lower", cmap="magma")
    ax.set_title(name, fontsize=9, fontweight="bold")
    ax.axis("off")

plt.tight_layout()
plt.savefig("results/plots/augmentation_examples.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nFeature extraction complete.")
print("All features cached to results/cache/")
print("Ready for model training.")